# Classification Baseline — EfficientNet-B3 on YOLO Crops

- 목적: YOLO crop → item_seq 단위 다중 클래스 분류
- 모델: EfficientNet-B3 (timm, pretrained, 300px)
- Bbox 소스: `yolo_pred` 기준 (GT crop 제외)
- 체크포인트 기준: val macro-F1 최대값
- **금지:** RandomHorizontalFlip, RandomVerticalFlip, RandomResizedCrop, 강한 Gaussian Blur
- **Split:** train crop manifest 80/20 내부 분할 (StratifiedGroupKFold, group=image_file)

## 0. Mount / Imports / Config

In [1]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

Mounted at /content/drive


In [2]:
!pip install timm scikit-learn -q

In [3]:
import gc
import json
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 50)

In [4]:
# ── 경로 상수 ──────────────────────────────────────────────────────────────────
DRIVE_BASE    = Path('/content/drive/MyDrive/Conference/Pillot')
CACHE_DIR     = DRIVE_BASE / 'cache'
CROP_DIR      = CACHE_DIR / 'crops'
CROP_PROG_DIR = CACHE_DIR / 'crop_progress'
YOLO_BBOX_DIR = CACHE_DIR / 'yolo_bbox'
LABEL_MAP_DIR = CACHE_DIR / 'label_map'
CKPT_DIR      = CACHE_DIR / 'classification' / 'jh_effb3_300'

TRAIN_MANIFEST_PATH = CROP_PROG_DIR / 'train_crop_manifest.csv'
TRAIN_YOLO_CSV      = YOLO_BBOX_DIR / 'train_yolo_pred.csv'
LABEL_MAP_PATH      = LABEL_MAP_DIR / 'item_seq_to_idx.json'

LOCAL_CROP_ROOT = Path('/content/crops_local')

# ── 하이퍼파라미터 ──────────────────────────────────────────────────────────────
INPUT_SIZE   = 300
BATCH_SIZE   = 32
NUM_EPOCHS   = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 1213
VAL_RATIO    = 0.2
SPLIT_SEED   = 42

# ── 재현성 ────────────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP_DIR.mkdir(parents=True, exist_ok=True)

device: cuda


## 0.5 이미지 로컬 복사 (Drive I/O 병목 방지)

Drive에서 직접 PNG를 읽으면 학습 중 ~19만 회 Drive 접근 → 속도 5~20× 저하.  

`LOCALIZE=True`로 한 번 복사 후, 이후 재시작 시에는 `LOCALIZE=False`로 스킵 가능.

In [5]:
LOCALIZE = True

if LOCALIZE:
    print('Copying crops from Drive to local SSD...')
    shutil.copytree(str(CROP_DIR), str(LOCAL_CROP_ROOT), dirs_exist_ok=True)
    print(f'Done. Local root: {LOCAL_CROP_ROOT}')
else:
    if not LOCAL_CROP_ROOT.exists():
        raise RuntimeError('LOCAL_CROP_ROOT does not exist. Set LOCALIZE=True and re-run.')
    print(f'Skipping copy. Using existing: {LOCAL_CROP_ROOT}')


def remap_crop_path(p: str) -> str:
    """drive/MyDrive/.../cache/crops/{split}/... → crops_local/{split}/..."""
    suffix = p.split('cache/crops/')[-1]
    return str(LOCAL_CROP_ROOT / suffix)

Copying crops from Drive to local SSD...
Done. Local root: /content/crops_local


## 1. Manifest 로드 및 Train/Val Split

- 단일 train crop manifest 로드 → 필터링 → StratifiedGroupKFold 80/20 분할
- group key: `image_file` (같은 이미지에서 나온 crop이 train/val에 섞이지 않도록)
- 분할 전 필터 순서: `yolo_pred` → `has_detection` → `label_map`

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def load_and_filter_manifest(manifest_path, yolo_csv_path):
    df = pd.read_csv(manifest_path)
    print(f'raw rows: {len(df):,}')

    non_yolo = (df['source_bbox'] != 'yolo_pred').sum()
    df = df[df['source_bbox'] == 'yolo_pred'].copy()
    print(f'  Dropped non-yolo_pred: {non_yolo:,}')

    if yolo_csv_path.exists():
        yolo_df = pd.read_csv(yolo_csv_path)[['image_file', 'has_detection']]
        df = df.merge(yolo_df, on='image_file', how='left')
        df['has_detection'] = df['has_detection'].fillna(True)
        no_det = int((df['has_detection'] == False).sum())
        df = df[df['has_detection'] == True].copy()
        print(f'  Dropped has_detection=False: {no_det:,}')

    oom = (~df['item_seq'].isin(label_map)).sum()
    df = df[df['item_seq'].isin(label_map)].copy()
    print(f'  Dropped out-of-label-map: {oom:,}')

    df['class_idx'] = df['item_seq'].map(label_map)
    df['crop_path'] = df['crop_path'].map(remap_crop_path)
    print(f'  Final: {len(df):,} rows, {df["item_seq"].nunique()} classes')
    return df.reset_index(drop=True)


all_df = load_and_filter_manifest(TRAIN_MANIFEST_PATH, TRAIN_YOLO_CSV)

sgkf = StratifiedGroupKFold(n_splits=round(1 / VAL_RATIO), shuffle=True, random_state=SPLIT_SEED)
train_idx, val_idx = next(sgkf.split(all_df, all_df['item_seq'], all_df['image_file']))

train_df = all_df.iloc[train_idx].reset_index(drop=True)
val_df   = all_df.iloc[val_idx].reset_index(drop=True)

leaked = set(train_df['image_file']) & set(val_df['image_file'])
assert len(leaked) == 0, f'image_file leakage: {len(leaked)} files'

print(f'\ntrain: {len(train_df):,} rows / val: {len(val_df):,} rows')
print(f'train drugs: {train_df["item_seq"].nunique()} / val drugs: {val_df["item_seq"].nunique()}')

print('\n--- train class distribution (top 10) ---')
display(train_df['item_seq'].value_counts().head(10))
print('\n--- val class distribution (top 10) ---')
display(val_df['item_seq'].value_counts().head(10))

## 3. Label Map

- train item_seq를 오름차순 정렬 → 결정론적 class_idx 부여
- 한 번 생성 후 JSON 고정 (train/val/inference 공유, 절대 재생성 안 함)

In [ ]:
if LABEL_MAP_PATH.exists():
    with open(LABEL_MAP_PATH) as f:
        label_map = {int(k): v for k, v in json.load(f).items()}
    print(f'Loaded existing label map: {len(label_map)} classes  ({LABEL_MAP_PATH})')
else:
    train_manifest_tmp = pd.read_csv(TRAIN_MANIFEST_PATH)
    sorted_classes = sorted(train_manifest_tmp['item_seq'].unique())
    label_map = {int(seq): idx for idx, seq in enumerate(sorted_classes)}
    with open(LABEL_MAP_PATH, 'w') as f:
        json.dump({str(k): v for k, v in label_map.items()}, f, indent=2)
    print(f'Created label map: {len(label_map)} classes  -> {LABEL_MAP_PATH}')
    del train_manifest_tmp

NUM_CLASSES = len(label_map)
idx_to_item_seq = {v: k for k, v in label_map.items()}
print(f'NUM_CLASSES = {NUM_CLASSES}')

## 5. Dataset 클래스

In [ ]:
class PillCropDataset(Dataset):
    def __init__(self, df, transform):
        self.records = df[['crop_path', 'class_idx', 'pad_fraction']].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row['crop_path']).convert('RGB')
        return self.transform(img), int(row['class_idx']), float(row['pad_fraction'])

## 6. Augmentation

**금지 목록:** `RandomHorizontalFlip`, `RandomVerticalFlip` (각인 방향 의미있음),  
`RandomResizedCrop` (bbox 정렬된 crop에서 강한 crop → 각인/분할선 잘릴 수 있음),  
강한 Gaussian Blur (OCR 보존 원칙).

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomAffine(
        degrees=(-15, 15),
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        interpolation=InterpolationMode.BILINEAR,
    ),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = PillCropDataset(train_df, train_transform)
val_ds   = PillCropDataset(val_df,   val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'train: {len(train_ds):,} samples  val: {len(val_ds):,} samples')

## 7. 모델

EfficientNet-B3 (timm native PyTorch, 300px input, ImageNet pretrained)

In [ ]:
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B3  |  params: {total_params/1e6:.1f}M  |  num_classes: {NUM_CLASSES}')

## 8. 학습 루프

- Optimizer: AdamW, Cosine Annealing LR
- Loss: CrossEntropyLoss (balanced subset이므로 Focal 불필요)
- AMP: T4 기준 ~2× 속도, OOM 방지
- Checkpoint: `best.pt` (val macro-F1 기준) + `latest.pt` (매 epoch, resume용)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

best_macro_f1 = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    train_correct = 0

    for imgs, labels, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [train]', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss    += loss.item() * len(labels)
        train_correct += (logits.argmax(1) == labels).sum().item()

    train_loss /= len(train_ds)
    train_acc   = train_correct / len(train_ds)

    # ── Val ──────────────────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels, all_pad_fracs = [], [], []

    with torch.no_grad():
        for imgs, labels, pad_fracs in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [val]', leave=False):
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                logits = model(imgs.to(device))
            preds = logits.argmax(1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
            all_pad_fracs.extend(pad_fracs.tolist())

    val_macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    val_acc      = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)

    scheduler.step()

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS}  '
          f'train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  '
          f'val_macro_f1={val_macro_f1:.4f}  val_acc={val_acc:.4f}')

    # ── Checkpoint ───────────────────────────────────────────────────────────
    torch.save(model.state_dict(), CKPT_DIR / 'latest.pt')

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        torch.save(model.state_dict(), CKPT_DIR / 'best.pt')
        meta = {
            'epoch': epoch,
            'val_macro_f1': round(val_macro_f1, 6),
            'val_accuracy': round(val_acc, 6),
            'train_loss': round(train_loss, 6),
        }
        with open(CKPT_DIR / 'best_meta.json', 'w') as f:
            json.dump(meta, f, indent=2)
        print(f'  → best checkpoint saved  (macro_f1={val_macro_f1:.4f})')

print(f'\nTraining done. Best val macro-F1: {best_macro_f1:.4f}')
print(f'Checkpoints: {CKPT_DIR}')

## 9. 평가 (best.pt 기준)

Macro-F1 / per-class recall / Confusion Matrix

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best.pt', map_location=device))
model.eval()

all_preds, all_labels, all_pad_fracs = [], [], []
with torch.no_grad():
    for imgs, labels, pad_fracs in tqdm(val_loader, desc='Eval'):
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            logits = model(imgs.to(device))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())
        all_pad_fracs.extend(pad_fracs.tolist())

macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f'Val macro-F1 : {macro_f1:.4f}')
print(f'Val accuracy : {accuracy:.4f}')

In [ ]:
target_names = [str(idx_to_item_seq[i]) for i in range(NUM_CLASSES)]
report = classification_report(
    all_labels, all_preds,
    labels=list(range(NUM_CLASSES)),
    target_names=target_names,
    output_dict=True,
    zero_division=0,
)
per_class_df = pd.DataFrame(report).T.iloc[:NUM_CLASSES][['precision', 'recall', 'f1-score', 'support']]
per_class_df = per_class_df.sort_values('recall')

print('--- Per-class Recall (worst 20) ---')
display(per_class_df.head(20))

In [ ]:
cm = confusion_matrix(all_labels, all_preds, normalize='true')

fig_size = max(12, NUM_CLASSES // 3)
plt.figure(figsize=(fig_size, fig_size))
plt.imshow(cm, aspect='auto', cmap='Blues', vmin=0, vmax=1)
plt.colorbar(fraction=0.03)
plt.title(f'Normalized Confusion Matrix (val)  macro-F1={macro_f1:.4f}')
plt.xlabel('Predicted class_idx')
plt.ylabel('True class_idx')
plt.tight_layout()
plt.show()

## 10. Failure Analysis — pad_fraction

`pad_fraction`이 높은 샘플이 오분류에 몰리면 classifier 문제가 아닌 **crop framing 문제**일 수 있습니다.

In [ ]:
result_df = pd.DataFrame({
    'true_label':   all_labels,
    'pred_label':   all_preds,
    'pad_fraction': all_pad_fracs,
})
result_df['correct'] = result_df['true_label'] == result_df['pred_label']

print('Val pad_fraction distribution:')
display(result_df['pad_fraction'].describe())

if result_df['pad_fraction'].max() < 1e-6:
    print('\npad_fraction이 모두 0 → 이 zip에서는 pad_fraction 분석 불가.')
    print('다른 zip(edge-heavy 샘플 포함)에서 crop 생성 후 재분석하세요.')
else:
    bins = [0.0, 0.05, 0.1, 0.2, 0.5, 1.01]
    result_df['pad_bin'] = pd.cut(result_df['pad_fraction'], bins=bins, right=False)
    pad_analysis = result_df.groupby('pad_bin')['correct'].agg(['mean', 'count'])
    pad_analysis.columns = ['accuracy', 'count']
    print('\n--- Accuracy by pad_fraction bin ---')
    display(pad_analysis)

## 11. (Optional) 224 vs 300 비교

`COMPARE_224 = True`로 바꾸면 동일 조건으로 224px 모델을 추가 학습합니다.

In [ ]:
COMPARE_224 = False

if COMPARE_224:
    INPUT_SIZE_224 = 224
    CKPT_DIR_224   = CACHE_DIR / 'classification' / 'jh_effb3_224'
    CKPT_DIR_224.mkdir(parents=True, exist_ok=True)

    train_transform_224 = transforms.Compose([
        transforms.Resize((INPUT_SIZE_224, INPUT_SIZE_224), interpolation=InterpolationMode.BILINEAR),
        transforms.RandomAffine(degrees=(-15, 15), translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_transform_224 = transforms.Compose([
        transforms.Resize((INPUT_SIZE_224, INPUT_SIZE_224), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    # Section 8 학습 루프 동일하게 재사용 (CKPT_DIR → CKPT_DIR_224 교체)
    print('COMPARE_224=True: Section 8 루프를 복사하고 CKPT_DIR를 CKPT_DIR_224로 교체하세요.')
else:
    print('COMPARE_224=False: skipped.')